# Notebook 04: Fundamentos Matemáticos

**TFM — Estimación de Pose 6-DoF mediante Transformers y Modelos de Difusión**

1. **Grupos de Lie:** SE(3) y SO(3) — exp/log maps
2. **Representaciones de rotación:** Quaterniones, 6D continua
3. **Score matching y difusión:** Toy example en 2D
4. **Conexión con el TFM**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import torch
import torch.nn.functional as F
import sys; sys.path.insert(0, '..')
from src.utils.lie_groups import (
    so3_exp, so3_log, so3_hat,
    se3_exp, se3_log, se3_inverse, se3_compose,
    pose_from_Rt, pose_to_Rt,
    geodesic_distance_SO3, geodesic_distance_SE3,
)
from src.utils.rotations import (
    quat_to_matrix, matrix_to_quat,
    euler_to_matrix, matrix_to_euler,
    matrix_to_6d, sixd_to_matrix,
    sixd_to_matrix_torch, matrix_to_6d_torch,
)
plt.rcParams['figure.figsize'] = (10, 6)
np.random.seed(42)

## 1. SO(3): El Grupo de Rotaciones 3D
SO(3) = {R in R3x3 : R^T R = I, det(R) = 1}

In [ ]:
axis = np.array([0, 0, 1])
angle = np.pi / 4
omega = axis * angle
R = so3_exp(omega)
print(f'Rotacion {np.degrees(angle):.0f} grados eje Z:')
print(np.round(R, 4))
print(f'R^T R = I: {np.allclose(R.T @ R, np.eye(3))}')
print(f'det(R) = {np.linalg.det(R):.6f}')
omega_rec = so3_log(R)
print(f'Roundtrip error: {np.linalg.norm(omega - omega_rec):.2e}')

In [ ]:
fig = plt.figure(figsize=(14, 5))
colors = ['r', 'g', 'b']
labels = ['X', 'Y', 'Z']
for idx, (a, title) in enumerate([(0,'Identidad'),(45,'Rz(45)'),(90,'Rz(90)')]):
    ax = fig.add_subplot(1, 3, idx+1, projection='3d')
    Rp = so3_exp(np.array([0, 0, np.radians(a)]))
    for i in range(3):
        ax.quiver(0,0,0, *Rp[:,i], color=colors[i], arrow_length_ratio=0.1, linewidth=3, label=labels[i])
    ax.set_xlim([-1.2,1.2]); ax.set_ylim([-1.2,1.2]); ax.set_zlim([-1.2,1.2])
    ax.set_title(title); ax.legend(loc='upper left')
plt.suptitle('Rotaciones en SO(3)', fontsize=14)
plt.tight_layout(); plt.show()

## 2. SE(3): Transformaciones Rígidas (Pose 6-DoF)

In [ ]:
R_obj = so3_exp(np.array([0.3, -0.5, 0.7]))
t_obj = np.array([0.15, -0.08, 0.45])
T_obj = pose_from_Rt(R_obj, t_obj)
print('Pose T in SE(3):'); print(np.round(T_obj, 4))
xi = se3_log(T_obj)
print(f'Twist: v={xi[:3].round(4)}, w={xi[3:].round(4)}')
T_gt = T_obj
noise = np.random.randn(6) * 0.05
T_est = se3_compose(T_gt, se3_exp(noise))
rot_err, trans_err = geodesic_distance_SE3(T_gt, T_est)
print(f'Error rot: {np.degrees(rot_err):.2f} deg, trans: {trans_err*1000:.2f} mm')

## 3. Representaciones de Rotación y 6D Continua (Zhou et al. CVPR 2019)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
R_start = np.eye(3)
R_end = so3_exp(np.array([np.pi, 0.5, 0.3]))
n_steps = 100
euler_v, quat_v, sixd_v, omega_v = [], [], [], []
w0, w1 = so3_log(R_start), so3_log(R_end)
for i in range(n_steps):
    t = i/(n_steps-1)
    w = (1-t)*w0 + t*w1
    Ri = so3_exp(w)
    euler_v.append(matrix_to_euler(Ri))
    quat_v.append(matrix_to_quat(Ri))
    sixd_v.append(matrix_to_6d(Ri))
    omega_v.append(w)
euler_v, quat_v, sixd_v, omega_v = map(np.array, [euler_v, quat_v, sixd_v, omega_v])
axes[0,0].plot(np.degrees(euler_v)); axes[0,0].set_title('Euler (discontinuo!)'); axes[0,0].legend(['R','P','Y'])
axes[0,1].plot(quat_v); axes[0,1].set_title('Quaternion'); axes[0,1].legend(['w','x','y','z'])
axes[1,0].plot(sixd_v); axes[1,0].set_title('6D Continua (suave)'); axes[1,0].legend([f'd{i}' for i in range(6)])
axes[1,1].plot(omega_v); axes[1,1].set_title('Axis-angle'); axes[1,1].legend(['wx','wy','wz'])
for ax in axes.flat: ax.set_xlabel('Step'); ax.grid(True, alpha=0.3)
plt.suptitle('Continuidad de representaciones durante interpolacion', fontsize=14)
plt.tight_layout(); plt.show()

In [ ]:
# 6D regresion diferenciable con PyTorch
R_target = torch.tensor(so3_exp(np.array([0.5, -0.3, 0.8])), dtype=torch.float32)
pred_6d = torch.randn(6, requires_grad=True)
opt = torch.optim.Adam([pred_6d], lr=0.05)
losses = []
for step in range(200):
    opt.zero_grad()
    R_pred = sixd_to_matrix_torch(pred_6d)
    loss = F.mse_loss(R_pred, R_target)
    loss.backward(); opt.step(); losses.append(loss.item())
R_final = sixd_to_matrix_torch(pred_6d).detach().numpy()
err = geodesic_distance_SO3(R_target.numpy(), R_final)
plt.semilogy(losses); plt.xlabel('Step'); plt.ylabel('MSE')
plt.title(f'6D Regression Convergence (final error: {np.degrees(err):.4f} deg)')
plt.grid(True, alpha=0.3); plt.show()

## 4. Score Matching y Difusión — Toy Example 2D
Forward: dx = f(x,t)dt + g(t)dw
Reverse: dx = [f - g^2 nabla_x log p_t(x)]dt + g dw_bar

In [ ]:
def sample_target(n=2000):
    centers = np.array([[2,2],[-2,2],[2,-2],[-2,-2]])
    return np.array([centers[np.random.randint(4)] + np.random.randn(2)*0.3 for _ in range(n)])

data = sample_target(2000)
sigmas = [0.0, 0.5, 1.0, 2.0, 4.0]
fig, axes = plt.subplots(1, 5, figsize=(20, 4))
for i, s in enumerate(sigmas):
    noisy = data + s*np.random.randn(*data.shape) if s > 0 else data
    axes[i].scatter(noisy[:,0], noisy[:,1], s=1, alpha=0.5, c='#0098CD')
    axes[i].set_xlim(-8,8); axes[i].set_ylim(-8,8); axes[i].set_title(f's={s}'); axes[i].set_aspect('equal')
plt.suptitle('Forward Diffusion: destruccion gradual de estructura', fontsize=14)
plt.tight_layout(); plt.show()

In [ ]:
def score_fn(x, centers=np.array([[2,2],[-2,2],[2,-2],[-2,-2]]), sigma=0.5):
    scores = np.zeros_like(x); weights = np.zeros(len(x))
    for c in centers:
        diff = x - c; w = np.exp(-0.5*np.sum(diff**2, axis=1)/sigma**2)
        scores -= (diff/sigma**2)*w[:,None]; weights += w
    return scores/(weights[:,None]+1e-8)

xx, yy = np.meshgrid(np.linspace(-5,5,25), np.linspace(-5,5,25))
grid = np.stack([xx.ravel(), yy.ravel()], axis=1)
sc = score_fn(grid)
fig, ax = plt.subplots(figsize=(8,8))
ax.quiver(grid[:,0], grid[:,1], sc[:,0], sc[:,1], color='#006C8F', alpha=0.7, scale=30)
ax.scatter(data[:,0], data[:,1], s=1, alpha=0.3, c='#0098CD')
centers = np.array([[2,2],[-2,2],[2,-2],[-2,-2]])
ax.scatter(centers[:,0], centers[:,1], s=200, c='red', marker='x', linewidths=3, label='Modos de agarre')
ax.set_title('Score Field nabla_x log p(x)'); ax.set_aspect('equal'); ax.legend(); plt.show()

In [ ]:
# Langevin dynamics sampling
def langevin(n=800, steps=300, eps=0.05):
    x = np.random.randn(n, 2)*4; traj = [x.copy()]
    for t in range(steps):
        x = x + (eps/2)*score_fn(x) + np.sqrt(eps)*np.random.randn(*x.shape)
        if t % 50 == 0: traj.append(x.copy())
    traj.append(x.copy()); return x, traj

samples, traj = langevin()
fig, axes = plt.subplots(1, len(traj), figsize=(4*len(traj), 4))
for i, pts in enumerate(traj):
    axes[i].scatter(pts[:,0], pts[:,1], s=1, alpha=0.5, c='#0098CD')
    axes[i].scatter(centers[:,0], centers[:,1], s=100, c='red', marker='x', linewidths=2)
    axes[i].set_xlim(-6,6); axes[i].set_ylim(-6,6); axes[i].set_aspect('equal')
    axes[i].set_title(f'Step {i*50}')
plt.suptitle('Reverse Diffusion via Langevin: ruido -> trayectorias de agarre', fontsize=14)
plt.tight_layout(); plt.show()
print('En Diffusion Policy: ruido gaussiano -> trayectorias de agarre multimodales')

In [ ]:
# Resumen de conceptos
concepts = [
    ('SE(3)/SO(3)', 'Representacion de poses 6-DoF', 'Lynch & Park 2017'),
    ('Exp map', 'Parametrizacion suave de rotaciones', 'Sola et al. 2018'),
    ('6D continua', 'Output de FoundationPose/GDR-Net', 'Zhou et al. CVPR 2019'),
    ('Score matching', 'Training de Diffusion Policy', 'Song & Ermon NeurIPS 2019'),
    ('Langevin', 'Sampling de trayectorias', 'Welling & Teh ICML 2011'),
    ('DDPM', 'Arquitectura Diffusion Policy', 'Ho et al. NeurIPS 2020'),
    ('Multi-head attn', 'Cross-attention en FoundationPose', 'Vaswani et al. 2017'),
]
print(f'{"Concepto":<18} {"Uso en TFM":<42} {"Referencia"}')
print('='*85)
for c, u, r in concepts: print(f'{c:<18} {u:<42} {r}')
print('\nTodos implementados y validados en este repositorio.')